In [0]:
#DimCustomer with SCD Type 2 (tracks history)
from pyspark.sql.functions import current_timestamp,lit,col
# First-time load: initialize the SCD Type 2 dimension
customers_scd=(
    spark.table("retailworks.silver.customers_clean")
     .withColumn("effective_date",current_timestamp())
    .withColumn("end_date",lit(None).cast("timestamp"))
    .withColumn("is_current",lit(True))
)
customers_scd.write.format("delta").mode("overwrite").saveAsTable("retailworks.gold.dim_customer")

In [0]:
#Applying an SCD Type 2 update (e.g., a customer moves city):
from delta.tables import DeltaTable

dim_customer=DeltaTable.forName(spark,"retailworks.gold.dim_customer")
# Simulate an incoming change: Alice Johnson moved from Chicago to Denver
updates=spark.createDataFrame([("C001","Alice Johnson","cust1@example.com","Denver","SMB")],
                              ["cust_id", "cust_name", "email", "city", "segment"])

#step-1:expire the old record
(
    dim_customer.alias("target")
    .merge(updates.alias("source"),"target.cust_id=source.cust_id AND target.is_current=true")
    .whenMatchedUpdate(
        condition="target.city<> source.city",
        set={"end_date":"current_timestamp()","is_current":"false"}
    )
    .execute()
)
#step-2:insert the new current record
new_records=updates.withColumn("effective_date",current_timestamp())\
                   .withColumn("end_date",lit(None).cast("timestamp"))\
                   .withColumn("is_current",lit(True))

new_records.write.format("delta").mode("append").saveAsTable("retailworks.gold.dim_customer")

In [0]:
#DimProduct (SCD Type 1 — simpler, for contrast)

products_dim=spark.table("retailworks.silver.products_clean")
products_dim.write.format("delta").mode("overwrite").saveAsTable("retailworks.gold.dim_product")

In [0]:
#DimDate

from pyspark.sql.functions import col,explode,sequence,to_date,year,month,dayofmonth,date_format,quarter
date_df=spark.sql("""
                  SELECT explode(sequence(to_date('2026-01-01'),to_date('2026-12-31'),interval 1 day)) as full_date""")
dim_date=(
    date_df
    .withColumn("date_id",date_format(col("full_date"),"yyyyMMdd").cast("int"))
    .withColumn("year",year(col("full_date")))
    .withColumn("month",month(col("full_date")))
    .withColumn("day",dayofmonth(col("full_date")))
    .withColumn("quarter",quarter(col("full_date")))
    .withColumn("month_name",date_format(col("full_date"),"MMM"))
    .withColumn("day_name",date_format(col("full_date"),"EEEE"))
)
dim_date.write.format("delta").mode("overwrite").saveAsTable("retailworks.gold.dim_date")


In [0]:
#FactSales (the grain: one row per order line — Transactional Fact Table)
from pyspark.sql.functions import date_format,col
fact_sales=(
    spark.table("retailworks.silver.orders_clean").alias("o")
    .join(spark.table("retailworks.gold.dim_customer").filter("is_current=true").alias("c"),"cust_id")
    .join(spark.table("retailworks.gold.dim_product").alias("p"),"prod_id")
    .withColumn("date_id",date_format(col("order_date"),"yyyyMMdd").cast("int"))
    .withColumn("total_amount",col("quantity")*col("unit_price"))
    .select(
        col("order_id"),
        col("cust_id"),
        col("prod_id"),
        col("date_id"),
        col("quantity"),
        col("unit_price"),
        col("total_amount")
    )
    )
fact_sales.write.format("delta").mode("overwrite").saveAsTable("retailworks.gold.fact_sales")

In [0]:
%sql
--Star Schema
--"What's total revenue by month and customer segment?" 
SELECT
    d.month_name,
    c.segment,
    SUM(f.total_amount) AS total_revenue,
    COUNT(DISTINCT f.order_id) AS order_count
FROM retailworks.gold.fact_sales f
JOIN retailworks.gold.dim_date d ON f.date_id=d.date_id
JOIN retailworks.gold.dim_customer C  ON F.cust_id=c.cust_id AND c.is_current=TRUE
GROUP BY d.month_name,c.segment
ORDER BY total_revenue DESC;

